# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> ⚠️ **Quy tắc:** Notebook chỉ để prototype. Sau khi chạy được ở đây, bạn phải **chuyển logic vào `src/multi_agent_research_lab/`** và pass tests. Các ô có `TODO(student)` là phần bạn phải tự viết.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [ ]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.errors import StudentTodoError
from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        if "analyst" in sys_lower:
            content = (
                "### Phân tích chuyên sâu RAG vs Fine-Tuning:\n"
                "- RAG tối ưu khi dữ liệu thay đổi liên tục và cần nguồn trích dẫn.\n"
                "- Fine-tuning tối ưu cho phong cách định dạng và tác vụ hẹp.\n"
                "- Cần kết hợp Hybrid để đạt độ chính xác cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "## Tổng Hợp Nghiên Cứu: RAG vs Fine-Tuning\n\n"
                "RAG và Fine-tuning là hai phương pháp bổ trợ then chốt [1]. "
                "RAG giúp giảm thiểu hallucination bằng cách truy xuất tài liệu ngoài [2], "
                "trong khi Fine-tuning định hình hành vi và giảm độ trễ [3].\n\n"
                "### Tài liệu tham khảo\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = f"Tổng quan câu trả lời cho: {user_prompt[:60]}"

        in_tok = max(10, len(system_prompt + user_prompt) // 4)
        out_tok = max(10, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tok, output_tokens=out_tok)


search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")


In [ ]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """Phân tích sources thành analysis_notes."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        if not state.sources:
            state.errors.append("DemoAnalyst: Không có sources")
            return state
        sys_prompt = "You are an AI analyst evaluating research evidence."
        user_prompt = f"Query: {state.request.query}\nNotes:\n{state.research_notes}"
        resp = self.llm_client.complete(sys_prompt, user_prompt)
        state.analysis_notes = resp.content
        state.agent_results.append(
            AgentResult(agent=AgentName.ANALYST, content=resp.content, metadata={"tokens": resp.output_tokens})
        )
        state.add_trace_event("analyst.done", {"tokens": resp.output_tokens})
        return state


class DemoWriterAgent:
    """Viết final_answer có trích dẫn nguồn."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        sys_prompt = "You are an expert technical writer synthesizing final report with citations."
        src_text = "\n".join(f"[{i}] {d.title} ({d.url})" for i, d in enumerate(state.sources, 1))
        user_prompt = f"Query: {state.request.query}\nAnalysis: {state.analysis_notes}\nSources:\n{src_text}"
        resp = self.llm_client.complete(sys_prompt, user_prompt)
        state.final_answer = resp.content
        state.agent_results.append(
            AgentResult(agent=AgentName.WRITER, content=resp.content, metadata={"tokens": resp.output_tokens})
        )
        state.add_trace_event("writer.done", {"tokens": resp.output_tokens})
        return state


state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)


In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        # TODO(student): Trả về MockLLMResponse hợp lý dựa trên system_prompt.
        # Gợi ý: phân nhánh theo vai trò (analyst / writer) xuất hiện trong system_prompt,
        # trả về nội dung giả lập khác nhau + ước lượng token (vd: len(prompt) // 4).
        raise StudentTodoError("TODO(student): implement MockLLMClient.complete")


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")

MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    if not state.sources:
        return "researcher"
    if state.sources and not state.analysis_notes:
        return "analyst"
    if state.analysis_notes and not state.final_answer:
        return "writer"
    return "done"


In [ ]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """TODO(student): phân tích sources thành analysis_notes."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        # TODO(student): Implement theo pattern của DemoResearcherAgent:
        # 1. Guard: nếu state.sources rỗng → append vào state.errors và return sớm.
        # 2. Gọi self.llm_client.complete(system_prompt="You are an analyst...", user_prompt=...)
        # 3. Ghi state.analysis_notes, append AgentResult(agent=AgentName.ANALYST, ...)
        # 4. Ghi trace event "analyst.done".
        raise StudentTodoError("TODO(student): implement DemoAnalystAgent.run")


class DemoWriterAgent:
    """TODO(student): viết final_answer có trích dẫn nguồn."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        # TODO(student):
        # 1. Dùng analysis_notes (fallback research_notes) làm ngữ cảnh.
        # 2. Gọi LLM để viết câu trả lời cuối cho state.request.audience.
        # 3. Bắt buộc kèm danh sách citation dạng [1] title (url) từ state.sources.
        # 4. Ghi state.final_answer + AgentResult(agent=AgentName.WRITER, ...) + trace.
        raise StudentTodoError("TODO(student): implement DemoWriterAgent.run")


# Smoke test agent mẫu
state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)

## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [ ]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    # Guard chống vòng lặp vô hạn — GIỮ NGUYÊN dòng này
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    # TODO(student): Implement routing policy dựa trên các field còn thiếu:
    # - Chưa có sources        → 'researcher'
    # - Có sources, chưa có analysis_notes → 'analyst'
    # - Có analysis_notes, chưa có final_answer → 'writer'
    # - Đã có final_answer     → 'done'
    # Nâng cao: nếu state.errors không rỗng thì xử lý fallback thế nào?
    raise StudentTodoError("TODO(student): implement demo_supervisor_route")

from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)
    resp = MockLLMClient().complete(
        "You are an AI assistant answering directly.",
        f"Answer query: {query_text}",
    )
    state.final_answer = resp.content
    state.record_route("single_agent")
    return state


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    if not state.sources or not state.final_answer:
        return 0.0
    matched = 0
    for idx, s in enumerate(state.sources, start=1):
        if f"[{idx}]" in state.final_answer or s.title.lower() in state.final_answer.lower():
            matched += 1
    return round(matched / len(state.sources), 2)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"

results: list[BenchmarkMetrics] = []
for run_name, runner in [
    ("single_agent", run_single_agent),
    ("multi_agent", run_demo_workflow),
]:
    st, metrics = run_benchmark(run_name, demo_query, runner)
    metrics.citation_coverage = compute_citation_coverage(st)
    results.append(metrics)

print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
for m in results:
    print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")


In [ ]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


try:
    final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
    print("Route history:", final_state.route_history)
    print("\n=== FINAL ANSWER ===\n")
    print(final_state.final_answer)
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")
    print("→ Quay lại các ô trên, implement xong rồi chạy lại ô này.")

## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh. Baseline single-agent (1 lần gọi LLM, không search) **bạn tự viết**.

In [ ]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    # TODO(student):
    # 1. Tạo ResearchState từ query_text.
    # 2. Gọi MockLLMClient().complete() một lần duy nhất → gán state.final_answer.
    # 3. Return state. (So sánh chất lượng/citation với bản multi-agent!)
    raise StudentTodoError("TODO(student): implement run_single_agent baseline")


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    # TODO(student): Đếm số source có title/url xuất hiện trong state.final_answer,
    # chia cho tổng số sources (trả 0.0 nếu không có sources hoặc answer).
    raise StudentTodoError("TODO(student): implement compute_citation_coverage")


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"

try:
    results: list[BenchmarkMetrics] = []
    for run_name, runner in [
        ("single_agent", run_single_agent),
        ("multi_agent", run_demo_workflow),
    ]:
        st, metrics = run_benchmark(run_name, demo_query, runner)
        metrics.citation_coverage = compute_citation_coverage(st)
        results.append(metrics)

    print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
    for m in results:
        print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")

## 7. Next Steps — chuyển sang `src/`

Khi notebook chạy end-to-end, chuyển logic vào code chính thức:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` |
| `MockSearchClient` → provider thật | `services/search_client.py` |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py` |

Sau đó verify:
```bash
make lint && make test
python -m multi_agent_research_lab.cli run --query "..."
bash scripts/check_todos.sh   # đảm bảo không còn TODO trong src/
```